# Simulation for doctor agent 1.0 with full fine-tuning

Files needed:
*   Base doctor agent that was saved locally
*   Generated patients (from mistral_7B and llama2_7B)


Unzip generated patients

In [ ]:
!unzip ./mistral_7B-20250412T174846Z-001.zip
!unzip ./llama2_7B-20250412T175220Z-001.zip

Archive:  ./mistral_7B-20250412T174846Z-001.zip
  inflating: mistral_7B/patients_buddchiari_syndrome.json  
  inflating: mistral_7B/patients_obstructive_cardiomyopathy.json  
  inflating: mistral_7B/patients_wegeners_granulomatosis.json  
  inflating: mistral_7B/patients_subdural_haemorrhage.json  
  inflating: mistral_7B/patients_chronic_hypotension.json  
  inflating: mistral_7B/patients_thrombotic.json  
  inflating: mistral_7B/patients_varicose_veins.json  
  inflating: mistral_7B/patients_febrile_mucocutaneous_lymph_node_syndrome_mcls.json  
  inflating: mistral_7B/patients_cardiovascular_disease.json  
  inflating: mistral_7B/patients_atrial_fibrillation_and_flutter.json  
  inflating: mistral_7B/patients_aortic_aneurysm.json  
  inflating: mistral_7B/patients_cardiac_hypertrophy.json  
  inflating: mistral_7B/patients_hypertensive_renal_disease.json  
  inflating: mistral_7B/patients_vein_thrombosis.json  
  inflating: mistral_7B/patients_lymphangitis.json  
  inflating: mistral

Edit the following path to match the paths to the patients generated with mistral and llama2:

In [ ]:
folder_path_mistral = './mistral_7B/'
folder_path_llama2 = './llama2_7B/'

Read all patients generated


In [ ]:
import os
import json

folder_path = folder_path_mistral
json_contents = []

for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            content = json.load(f)
            json_contents.append(content)  # Append the parsed JSON to your list

print(len(json_contents))

folder_path = folder_path_llama2
json_contents2 = []

for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            content = json.load(f)
            json_contents2.append(content)  # Append the parsed JSON to your list

print(len(json_contents2))

102
29


Data handling

In [ ]:
import pandas as pd

df = pd.DataFrame()

for i in range(len(json_contents)):
    json_file = json_contents[i]
    for j in range(len(json_file)):
        json_file_elem = json_file[j]
        patients_df = pd.DataFrame(json_file_elem["patients"])
        df = pd.concat([df, patients_df], ignore_index=True)

df2 = pd.DataFrame()

for i in range(len(json_contents2)):
    json_file = json_contents2[i]
    for j in range(len(json_file)):
        json_file_elem = json_file[j]
        patients_df = pd.DataFrame(json_file_elem["patients"])
        df2 = pd.concat([df2, patients_df], ignore_index=True)


total_df = pd.concat([df, df2], ignore_index=True)
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5473 entries, 0 to 5472
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Full name              5473 non-null   object
 1   Age                    5473 non-null   object
 2   Gender                 5473 non-null   object
 3   Symptoms               5473 non-null   object
 4   Symptom duration       5473 non-null   object
 5   Recent travel history  5473 non-null   object
 6   Medical history        5473 non-null   object
 7   Mixed diseases         5473 non-null   object
 8   Correct disease        5473 non-null   object
 9   Diagnosis cause        5473 non-null   object
 10  patient_number         5473 non-null   object
dtypes: object(11)
memory usage: 470.5+ KB


In [ ]:
import pandas as pd
import ast  # for safely evaluating strings that represent lists

# Define transformation functions
def build_question(row):
    return (
        f"This patient is a {row['Age']} year old {row['Gender']} presenting with the following symptoms: "
        f"{row['Symptoms']}. Symptoms have lasted for {row['Symptom duration']}. "
        f"Recent travel history: {row['Recent travel history']}. "
        f"Medical history: {row['Medical history']}. "
        "What is the disease of this patient?"
    )

def build_options(diseases):
    return {chr(65 + i): disease for i, disease in enumerate(diseases)}

def get_answer_idx(options, correct_disease):
    for key, val in options.items():
        if val == correct_disease:
            return key
    return None

# Apply transformations
total_df["question"] = total_df.apply(build_question, axis=1)
total_df["options"] = total_df["Mixed diseases"].apply(build_options)
total_df["answer"] = total_df["Correct disease"]
total_df["answer_idx"] = total_df.apply(lambda row: get_answer_idx(row["options"], row["answer"]), axis=1)

# Keep only the required columns
final_df = total_df[["question", "options", "answer", "answer_idx"]].copy()
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5473 entries, 0 to 5472
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    5473 non-null   object
 1   options     5473 non-null   object
 2   answer      5473 non-null   object
 3   answer_idx  5473 non-null   object
dtypes: object(4)
memory usage: 171.2+ KB


In [ ]:
final_df.head(10)
final_df["answer_idx"].value_counts(normalize=True)

,proportion
answer_idx,
C,0.210122
B,0.205737
A,0.197698
D,0.193495
E,0.192947


In [ ]:
! pip3 install transformers datasets torch accelerate evaluate

# Now let's get serious

Load model and tokenizer

In [ ]:
!unzip ./epoch3.zip  # Unzip zipped model

Archive:  ./epoch3.zip
   creating: lr_5e-05/checkpoint-94551/
  inflating: lr_5e-05/checkpoint-94551/optimizer.pt  
  inflating: lr_5e-05/checkpoint-94551/tokenizer_config.json  
  inflating: lr_5e-05/checkpoint-94551/model.safetensors  
  inflating: lr_5e-05/checkpoint-94551/vocab.txt  
  inflating: lr_5e-05/checkpoint-94551/training_args.bin  
  inflating: lr_5e-05/checkpoint-94551/rng_state.pth  
  inflating: lr_5e-05/checkpoint-94551/scheduler.pt  
  inflating: lr_5e-05/checkpoint-94551/special_tokens_map.json  
  inflating: lr_5e-05/checkpoint-94551/scaler.pt  
  inflating: lr_5e-05/checkpoint-94551/config.json  
  inflating: lr_5e-05/checkpoint-94551/trainer_state.json  
  inflating: lr_5e-05/checkpoint-94551/tokenizer.json  


Edit the model_path variable to match the path of your pre-trained model:

In [ ]:
from transformers import BertTokenizer, BertForMultipleChoice

model_path = "./lr_5e-05/checkpoint-94551/"

tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForMultipleChoice.from_pretrained(model_path)

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at ./lr_5e-05/checkpoint-94551/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import AutoTokenizer
from tqdm import tqdm

def preprocess_medqa(examples):
    """Tokenize question and choices separately for multiple-choice classification"""
    inputs = {"input_ids": [], "attention_mask": [], "token_type_ids": [], "labels": []}

    for example in tqdm(examples, total=len(examples)):
    #for i in range(len(examples["question"])):  # Process each example individually
        question = example["question"]
        options = example["options"] # Dictionary {'A': 'Ampicillin', 'B': 'Ceftriaxone', ...}
        correct_answer = example["answer_idx"]  # Single letter ('A', 'B', ...)

        # Ensure consistent option order (sort by key)
        option_keys = sorted(options.keys())
        option_values = [options[key] for key in option_keys]  # List of answer choices

        # Convert correct answer letter to index
        if correct_answer in option_keys:
            label = option_keys.index(correct_answer)  # Map the letter to index (0, 1, ...)
        else:
            raise ValueError(f"Unexpected answer key: {correct_answer} in {option_keys}")

        # Tokenize question-answer pairs
        encoding = tokenizer(
            [question] * len(option_values),  # Repeat the question for each choice
            option_values,  # List of choices
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        # Append results to inputs dictionary
        inputs["input_ids"].append(encoding["input_ids"].squeeze(0))  # Remove extra batch dimension
        inputs["attention_mask"].append(encoding["attention_mask"].squeeze(0))
        inputs["token_type_ids"].append(encoding.get("token_type_ids", None).squeeze(0) if "token_type_ids" in encoding else None)
        inputs["labels"].append(label)  # Correct answer index

    return inputs

In [ ]:
import torch
from datasets import Dataset, DatasetDict

all_data = Dataset.from_pandas(final_df)
all_data_preprocessed = Dataset.from_dict(preprocess_medqa(all_data))

# Make sure the dataset is shuffled
dataset = all_data_preprocessed.shuffle(seed=42)

# Split into train (80%) and temp (20%)
train_testvalid = dataset.train_test_split(test_size=0.2, seed=42)

# Then split the 20% temp into valid and test (50/50 of 20% = 10% each)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

# Combine splits into a DatasetDict
dataset_dict = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})

train_dataset = dataset_dict["train"]
valid_dataset = dataset_dict["validation"]
test_dataset = dataset_dict["test"]

100%|██████████| 5473/5473 [01:01<00:00, 89.56it/s]


In [ ]:
# Verifying if we are working on the GPU
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Number of GPUs available
print(torch.cuda.get_device_name(0))  # GPU name
model.to("cuda")
print(next(model.parameters()).device)  # Should return: cuda:0

True
1
Tesla T4
cuda:0


In [ ]:
import evaluate

# Load metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    preds = predictions.argmax(axis=1)
    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    return {"accuracy": accuracy["accuracy"]}

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    run_name="doctor_agent_1_0",
    output_dir="./",
    evaluation_strategy="epoch",     # Change from "epoch"
    save_strategy="epoch",
    learning_rate=2e-5,
    fp16=True,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,               # Logs loss every 50 steps
    load_best_model_at_end=True,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  # Pass the metric function
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-18-953915ae8d62>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.100500,0.136430,0.957952
2,0.073300,0.115755,0.974406
3,0.023000,0.093055,0.981718
4,0.023900,0.077533,0.983547
5,0.009900,0.152397,0.978062
6,0.001400,0.154277,0.981718
7,0.000500,0.139565,0.981718
8,0.000000,0.169023,0.978062
9,0.000000,0.157615,0.981718
10,0.000500,0.144759,0.981718


TrainOutput(global_step=10950, training_loss=0.039229495672330465, metrics={'train_runtime': 5571.8817, 'train_samples_per_second': 7.857, 'train_steps_per_second': 1.965, 'total_flos': 5.75944928959488e+16, 'train_loss': 0.039229495672330465, 'epoch': 10.0})

In [ ]:
print(trainer.state.best_model_checkpoint)

./medical_agent_simu/checkpoint-4380


Save model

In [ ]:
model.save_pretrained("models/")
tokenizer.save_pretrained("models/")
print("Model saved")

Model saved


In [ ]:
!zip -r ./model_after_simul.zip ./models/*

In [ ]:
from google.colab import files
files.download('./model_after_simul.zip')

Evaluation on test set

In [ ]:
results = trainer.evaluate(test_dataset)
with open("doctor_agent_1_0.csv", "w") as f:
    for key, value in results.items():
        f.write(f"{key}: {value}\n")
print(results)

{'eval_loss': 0.050429817289114, 'eval_accuracy': 0.9854014598540146, 'eval_runtime': 21.9158, 'eval_samples_per_second': 25.005, 'eval_steps_per_second': 6.251, 'epoch': 10.0}
